In [1]:
# DICES speech file extraction

In [2]:
# imports
import os
import re
import pandas as pd
import numpy as np

In [3]:
# load Tesserae texts
book1t = 'ovid.metamorphoses.part.1.tess.txt'
book2t = 'ovid.metamorphoses.part.2.tess.txt'
book3t = 'ovid.metamorphoses.part.3.tess.txt'
book4t = 'ovid.metamorphoses.part.4.tess.txt'
book5t = 'ovid.metamorphoses.part.5.tess.txt'
book6t = 'ovid.metamorphoses.part.6.tess.txt'
book7t = 'ovid.metamorphoses.part.7.tess.txt'
book8t = 'ovid.metamorphoses.part.8.tess.txt'
book9t = 'ovid.metamorphoses.part.9.tess.txt'
book10t = 'ovid.metamorphoses.part.10.tess.txt'
book11t = 'ovid.metamorphoses.part.11.tess.txt'
book12t = 'ovid.metamorphoses.part.12.tess.txt'
book13t = 'ovid.metamorphoses.part.13.tess.txt'
book14t = 'ovid.metamorphoses.part.14.tess.txt'
book15t = 'ovid.metamorphoses.part.15.tess.txt'

In [4]:
# function to read the csv file and process the 'Locus' column
def process_locus_csv(csv_file):
    df = pd.read_csv(csv_file)
    locus_data = df['Locus'].tolist()
    return locus_data

In [5]:
# function to pull out book, start line, and end line for each speech
def parse_string_list(strings):
    # initialize empty lists for book, start, and end
    x_list = []
    y_list = []
    z_list = []
    
    # iterate through each string in the input list
    for s in strings:
        # pplit the string into the parts book.start and book.end
        xy, xz = s.split('-')
        x, y = xy.split('.')
        _, z = xz.split('.')
        
        # remove any whitespace, convert to integers, and append to respective lists
        x_list.append(int(x.strip()))
        y_list.append(int(y.strip()))
        z_list.append(int(z.strip()))
    
    return x_list, y_list, z_list

In [6]:
# speech references

# 100 line sheet
#locus_data = process_locus_csv('DICES Data for Met Speakers.csv')
#book, start, end = parse_string_list(locus_data)

# 50 line sheet 
locus_data_50 = process_locus_csv('DICES Data for Met Speakers 50 lines.csv')
book, start, end = parse_string_list(locus_data_50)

In [7]:
# function to find the lines within the specified range in the text file
def extract_lines(file_path, start_line, end_line):
    extracted_lines = []
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            match = re.match(r'<ov\. met\. (\d+)\.(\d+)>', line)
            if match:
                book_num = int(match.group(1))
                line_num = int(match.group(2))
                if start_line <= line_num <= end_line:
                    extracted_lines.append(line)
    return extracted_lines

In [8]:
# sample output from extract_lines function
extract_lines(book8t,550,559)

['<ov. met. 8.550>\timbre tumens. "Succede meis," ait "inclite, tectis,\n',
 '<ov. met. 8.551>\tCecropida, nec te committe rapacibus undis:\n',
 '<ov. met. 8.552>\tferre trabes solidas obliquaque volvere magno\n',
 '<ov. met. 8.553>\tmurmure saxa solent. Vidi contermina ripae\n',
 '<ov. met. 8.554>\tcum gregibus stabula alta trahi: nec fortibus illic\n',
 '<ov. met. 8.555>\tprofuit armentis, nec equis velocibus esse.\n',
 '<ov. met. 8.556>\tMulta quoque hic torrens nivibus de monte solutis\n',
 '<ov. met. 8.557>\tcorpora turbineo iuvenalia flumine mersit.\n',
 '<ov. met. 8.558>\tTutior est requies, solito dum flumina currant\n',
 '<ov. met. 8.559>\tlimite, dum tenues capiat suus alveus undas."\n']

In [9]:
# function for making speech files and saving descriptive names
def make_speech_file(file_input, book, book_ref, index):
    a = start[index]
    b = end[index]
    c = extract_lines(file_input, a, b)
    # list of references (for file labels)
    ref = f"{book_num}.{start[index]}-{book_num}.{end[index]}"
    book_ref.append(ref)
    # list of raw speech texts
    book.append(c)

In [10]:
# main loop for speech extraction 
book_speeches = {i: [] for i in range(1, 16)}
book_refs = {i: [] for i in range(1, 16)}

# Map book number to source text
book_texts = {
    1: book1t,
    2: book2t,
    3: book3t,
    4: book4t,
    5: book5t,
    6: book6t,
    7: book7t,
    8: book8t,
    9: book9t,
    10: book10t,
    11: book11t,
    12: book12t,
    13: book13t,
    14: book14t,
    15: book15t,
}

for index, book_num in enumerate(book):
    make_speech_file(
        book_texts[book_num],
        book_speeches[book_num],
        book_refs[book_num],
        index
    )

In [11]:
# function for producing cleaned txt files of each speech (i.e., remove author/work tags from each line)
output_dir = "output_speeches_50_lines"
os.makedirs(output_dir, exist_ok=True)

def save_cleaned_speech (book, book_refs):
    pattern = re.compile(r'<[^>]+>')
    for index, speech in enumerate(book):
        with open(os.path.join(output_dir, f"{book_refs[index]}.txt"),"w") as f:
            for line in speech:
                cleaned_line = pattern.sub("", line)
                f.write(cleaned_line)


In [12]:
# output final files
for book_num in range(1, 16):
    save_cleaned_speech(
        book_speeches[book_num],
        book_refs[book_num]
    )